# 01 - Data Collection
Mengambil data dari World Bank API menggunakan library `wbgapi`.

**Indikator yang diambil:**
- GDP Growth (NY.GDP.MKTP.KD.ZG)
- Inflasi (FP.CPI.TOTL.ZG)
- Pengangguran (SL.UEM.TOTL.ZS)
- Pertumbuhan Populasi (SP.POP.GROW)
- Ekspor % GDP (NE.EXP.GNFS.ZS)
- Impor % GDP (NE.IMP.GNFS.ZS)
- FDI % GDP (BX.KLT.DINV.WD.GD.ZS)
- Nilai Tukar (PA.NUS.FCRF)

In [1]:
import wbgapi as wb
import pandas as pd
import os

# Indikator World Bank
INDICATORS = {
    'NY.GDP.MKTP.KD.ZG': 'GDP_Growth',
    'FP.CPI.TOTL.ZG':    'Inflation',
    'SL.UEM.TOTL.ZS':    'Unemployment',
    'SP.POP.GROW':       'Population_Growth',
    'NE.EXP.GNFS.ZS':    'Exports',
    'NE.IMP.GNFS.ZS':    'Imports',
    'BX.KLT.DINV.WD.GD.ZS': 'FDI',
    'PA.NUS.FCRF':       'Exchange_Rate'
}

COUNTRY = 'IDN'  # Indonesia
START_YEAR = 1991
END_YEAR = 2024

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Mengambil data dari World Bank API...')

Mengambil data dari World Bank API...


In [2]:
# Ambil setiap indikator
dfs = []

for code, name in INDICATORS.items():
    print(f'Mengambil: {name} ({code})')
    try:
        df = wb.data.DataFrame(
            code,
            economy=COUNTRY,
            time=range(START_YEAR, END_YEAR + 1)
        )
        df = df.T.reset_index()
        df.columns = ['Year', name]
        df['Year'] = df['Year'].str.replace('YR', '').astype(int)
        df.to_csv(f'../data/raw/{name}.csv', index=False)
        dfs.append(df)
        print(f'  -> {len(df)} baris, missing: {df[name].isna().sum()}')
    except Exception as e:
        print(f'  -> ERROR: {e}')

print('\nSemua indikator berhasil diambil!')

Mengambil: GDP_Growth (NY.GDP.MKTP.KD.ZG)


  -> 34 baris, missing: 0
Mengambil: Inflation (FP.CPI.TOTL.ZG)


  -> 34 baris, missing: 0
Mengambil: Unemployment (SL.UEM.TOTL.ZS)


  -> 34 baris, missing: 0
Mengambil: Population_Growth (SP.POP.GROW)


  -> 34 baris, missing: 0
Mengambil: Exports (NE.EXP.GNFS.ZS)


  -> 34 baris, missing: 0
Mengambil: Imports (NE.IMP.GNFS.ZS)


  -> 34 baris, missing: 0
Mengambil: FDI (BX.KLT.DINV.WD.GD.ZS)


  -> 34 baris, missing: 0
Mengambil: Exchange_Rate (PA.NUS.FCRF)


  -> 34 baris, missing: 0

Semua indikator berhasil diambil!


In [3]:
# Gabungkan semua indikator
from functools import reduce

dataset = reduce(lambda left, right: pd.merge(left, right, on='Year', how='outer'), dfs)
dataset = dataset.sort_values('Year').reset_index(drop=True)

# Simpan
dataset.to_csv('../data/processed/dataset_indonesia.csv', index=False)

print('Dataset gabungan:')
print(dataset.shape)
dataset.head(10)

Dataset gabungan:
(34, 9)


,Year,GDP_Growth,Inflation,Unemployment,Population_Growth,Exports,Imports,FDI,Exchange_Rate
0,1991,6.911983,9.419068,2.617,1.770137,28.351261,26.488304,1.270772,1950.317500
1,1992,6.497507,7.523510,2.734,1.733226,30.307464,27.119971,1.387989,2029.920833
2,1993,6.496408,9.671911,2.782,1.701584,26.754813,23.768572,1.268301,2087.103867
3,1994,7.539971,8.531998,4.366,1.685952,26.511428,25.365673,1.192252,2160.753675
4,1995,8.220007,9.420303,4.611,1.662785,26.312165,27.646425,2.150080,2248.607975
5,1996,7.818187,7.973281,4.861,1.645462,25.824552,26.440192,2.724198,2342.296292
6,1997,4.699879,6.226152,4.684,1.626308,27.859243,28.134616,2.167797,2909.380000
7,1998,-13.126725,58.451041,5.459,1.580718,52.968135,43.218058,-0.252290,10013.622500
8,1999,0.791126,20.477832,6.358,1.503171,35.514129,27.429784,-1.332574,7855.150000
9,2000,4.920068,3.688620,6.077,1.432440,40.977308,30.459567,-2.757440,8421.775000


In [4]:
# Cek missing values
print('Missing values per kolom:')
missing = dataset.isnull().sum()
missing_pct = (missing / len(dataset) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Persen (%)': missing_pct})

Missing values per kolom:

,Missing,Persen (%)
Year,0,0.0
GDP_Growth,0,0.0
Inflation,0,0.0
Unemployment,0,0.0
Population_Growth,0,0.0
Exports,0,0.0
Imports,0,0.0
FDI,0,0.0
Exchange_Rate,0,0.0


In [5]:
# Statistik deskriptif
dataset.describe().round(3)

,Year,GDP_Growth,Inflation,Unemployment,Population_Growth,Exports,Imports,FDI,Exchange_Rate
count,34.000,34.000,34.000,34.000,34.000,34.000,34.000,34.000,34.000
mean,2007.500,4.650,8.330,5.012,1.279,27.480,24.708,1.317,9413.840
std,9.958,3.639,9.660,1.544,0.299,7.220,5.018,1.320,4343.159
min,1991.000,-13.127,1.560,2.617,0.705,17.331,15.641,-2.757,1950.318
25%,1999.250,4.804,3.909,3.883,1.086,22.554,21.060,0.976,8460.615
50%,2007.500,5.059,6.379,4.562,1.285,26.320,24.851,1.763,9542.796
75%,2015.750,6.135,9.420,6.100,1.485,30.435,27.352,2.163,13362.707
max,2024.000,8.220,58.451,8.060,1.770,52.968,43.218,2.916,15855.448
